In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (12, 8),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import rateslib as rl
import QuantLib as ql

import datetime
import pytz

NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
LDN_tz = pytz.timezone("Europe/London") 
UTC_tz = pytz.timezone("UTC") 

import sys
sys.path.append("../")

In [2]:
from MDP.IRSwaptions.IRSwaptionMDP import IRSwaptionMDP
from Query.Base.query_resolution import resolve_query
from Query.IRSwaptions import IRSwaptionQuery, IRSwaptionStructure, IRSwaptionValue


In [3]:
mdp = IRSwaptionMDP(
    source="GSQUANT-QL",
    curve_source="ERIS_EOD_LIVE-QL_BASIC",
)

In [19]:
as_of = datetime.date(2026, 3, 9)
ctx = mdp.get_pricer(
    {
        "endpoint": "swaption_snapshot",
        "curve_name": "USD-SOFR-1D",
        "timestamp": as_of,
        "surface_type": "atmf_normal",
        "ignore_cache": False,
    }
)
ctx

IRSwaptionMarketContext(curve_name='USD-SOFR-1D', as_of_date=datetime.date(2026, 3, 9), curve=QLIRSwapCurve(_ql_curve_id='USD-SOFR-1D', _ql_curve_handle=<QuantLib.QuantLib.YieldTermStructureHandle; proxy of <Swig Object of type 'Handle< YieldTermStructure > *' at 0x00000258F49066F0> >, _ql_curve_index=<QuantLib.QuantLib.Sofr; proxy of <Swig Object of type 'ext::shared_ptr< Sofr > *' at 0x00000258F49058B0> >, _meta_data={'timestamp': datetime.date(2026, 3, 9)}), curve_handle=<QuantLib.QuantLib.YieldTermStructureHandle; proxy of <Swig Object of type 'Handle< YieldTermStructure > *' at 0x00000258F49066F0> >, swap_index=<QuantLib.QuantLib.Sofr; proxy of <Swig Object of type 'ext::shared_ptr< Sofr > *' at 0x00000258F49058B0> >, vol_handle=<QuantLib.QuantLib.SwaptionVolatilityStructureHandle; proxy of <Swig Object of type 'Handle< SwaptionVolatilityStructure > *' at 0x00000258F48DE5B0> >, pricing_engine=<QuantLib.QuantLib.BachelierSwaptionEngine; proxy of <Swig Object of type 'ext::shared_pt

In [20]:
q = IRSwaptionQuery(
    curve="USD-SOFR-1D",
	shorthand="1m10y",
	structure=IRSwaptionStructure.STRADDLE,
    strike="ATMF",
    structure_kwargs={"notional": 500_000_000},
)

q_eff = resolve_query(q, timestamp=as_of, pricer_or_curve=ctx)
package, weights = q_eff.resolve_package(pricer_or_curve=ctx)
vmap = q_eff.build_value_map(pricer_or_curve=ctx, package=package, risk_weights=weights)

In [21]:
float(vmap.apply(IRSwaptionValue.NVOL)), float(vmap.apply(IRSwaptionValue.FWD_PREM)), float(vmap.apply(IRSwaptionValue.SPOT_PREM)), float(vmap.apply(IRSwaptionValue.VEGA_01))

(78.09181161724713, 152.29910493623726, 151.8184348886212, 97205.0921502063)